# Сборный проект - 2

**Описание проекта**

Вы работаете в стартапе, который продаёт продукты питания. Нужно разобраться, как ведут себя пользователи вашего мобильного приложения.

**Цель проекта** - изучить воронку продаж и проанализировать результаты A/A/B-эксперимента.

**Описание данных**

Каждая запись в логе — это действие пользователя, или событие.
- `EventName` — название события;
- `DeviceIDHash` — уникальный идентификатор пользователя;
- `EventTimestamp` — время события;
- `ExpId` — номер эксперимента: 246 и 247 — контрольные группы, а 248 — экспериментальная.

**План работы**

- Шаг 1. Изучить общую информацию из датасета
- Шаг 2. Подготовка данных
- Шаг 3. Изучить и проверить данные
- Шаг 4. Изучить воронку событий
- Шаг 5. Изучить результаты эксперимента

## Шаг 1. Загрузка данных 

In [ ]:
import pandas as pd

from scipy import stats as st
import numpy as np
import math as mth

import seaborn as sns
from matplotlib import pyplot as plt
from plotly import graph_objects as go

import datetime as dt

In [ ]:
try:
    df = pd.read_csv('/datasets/logs_exp.csv', sep='\t')
except:
    df = pd.read_csv(r'..\Downloads\practicum_info\logs_exp.csv', sep='\t')
    
df.head()

In [ ]:
df.info()

**Вывод**

Загружен датасет, кол-во строк 244126, стобец с датой и временем `EventTimestamp` имеет некорректный тип данных

## Шаг 2. Подготовка данных

- Заменить названия столбцов;
- Проверить пропуски и типы данных;
- Добавить столбец даты и времени, а также отдельный столбец дат;

In [ ]:
# изменяем название столбцов для удобства 
df.columns = ['event', 'device_id', 'event_time', 'exp_id']

In [ ]:
# проверяем на наличие дубликатов
df.duplicated().sum()

Кол-во дубликатов 413 из 244126 строк, это 0.17% - незначителный процент, можем удалить их

In [ ]:
df = df.drop_duplicates()

In [ ]:
# проверяем кол-во пропусков
df.isna().sum()

In [ ]:
# проверяем тип данных
df.info()

В столбце `event_time` неправильный тип данных, приведем к корректному и добавим столбец с датой

In [ ]:
df['event_time'] = pd.to_datetime(df['event_time'], unit='s')
df['date'] = df['event_time'].astype('datetime64[D]')

In [ ]:
df.head()

In [ ]:
df.info()

**Вывод**

- Столбцы переименованы (для удобства)
- Были найдены дубликаты и из-за их незначительного кол-ва удалены 
- Не обнаружено пропусков
- Столбец `event_time` приведен к типу datetime64
- Добавлен новый столбец с датой
- Отсутствуют пересечения пользователей в группах

In [ ]:
df.groupby('device_id').agg({'exp_id': 'nunique'}).query('exp_id > 1').count()

Отсутствуют пользователи, которые участвовали в двух или более групп одновременно

## Шаг 3. Изучение и проверка данных

In [ ]:
print('Кол-во событий в логе:',df['event'].count())

In [ ]:
print('Кол-во пользователей в логе:', df['device_id'].nunique())

In [ ]:
print('Среднее кол-во событий на одного пользователя:', (df['event'].count()/df['device_id'].nunique()).round())

In [ ]:
df.groupby('device_id')['event'].count().describe()

In [ ]:
#визуализируем
plt.figure(figsize=(10, 7))
plt.boxplot(df.groupby('device_id')['event'].count())

plt.grid()
plt.ylabel('Кол-во пользователей')
plt.title('Кол-во событий на пользователя')

plt.show()

Видно, что есть большое количество выбросов. Рационально использовать медиану, а не среднее

In [ ]:
print("Медианное кол-во событий на одного пользователя:", df.groupby('device_id')['event'].count().median())

In [ ]:
df['dt'] = pd.to_datetime(df['date']).dt.date

In [ ]:
print('Минимальная дата:', df['dt'].min())
print('Максимальная дата:',df['dt'].max())
print('Период данных:', df['dt'].max()-df['dt'].min())

In [ ]:
# гистограмма по дате и времени
temp = df.groupby('dt').agg({'event':'count'}).reset_index()

plt.subplots(figsize=(10, 5))
sns.barplot(x='dt', y='event', data=temp, palette='flare')

plt.title('Распределение событий по времени')
plt.ylabel('Кол-во событий')
plt.xlabel('Дата')
plt.xticks(rotation=25)

plt.show()

По графику видно, что до 2019-07-31 количество событий очень мало. Т.е полные данные мы имеем с 2019-08-01 по 2019-08-07 включительно. Отбросим данные до  2019-08-01

In [ ]:
data = df.query('date >= "2019-08-01"')

In [ ]:
print('Кол-во событий до фильтрации:', len(df))
print('Кол-во событий после фильтрации:', len(data))
print('Процент потеряных событий после фильтрации:', round((len(df)-len(data))/len(df)*100,2),'%')

In [ ]:
print('Кол-во пользователей до фильтрации:', df['device_id'].nunique())
print('Кол-во пользователей после фильтрации:', data['device_id'].nunique())
print('Процент потеряных пользователей после фильтрации:', \
      round((df['device_id'].nunique()-data['device_id'].nunique())/df['device_id'].nunique()*100,2),'%')

После фильтрации новый датасет уменьшился на 1.16 %, а кол-во уникальных пользователей на 0.23 %

Проверяем, есть ли пользователи из всех трёх экспериментальных групп

In [ ]:
#старый датасет
df.groupby('exp_id').agg({'device_id':'nunique'})

In [ ]:
#новый датасет
data.groupby('exp_id').agg({'device_id':'nunique'})

В каждой группе есть пользователи, также не заметно большой потери между датасетами 

**Вывод**

- Представлены данный за период с 2019-07-25 до 2019-08-07
- Всего было зарегитрировано 243713 событий и 7551 пользователь
- Распределение событий показало, что тестирование началось 2019-08-01 (чтобы корректно оценить поведение пользователей были удалены события до этой даты)
- Доля отфильтрованных данных составила 1,16% от общего кол-ва логов и 0,23% от общего кол-ва пользователей
- Пользователи поделены на три примерно равнозначные группы

Делаем вывод, что тестирование можно считать корректным

## Шаг 4. Изучение воронки событий

Смотрим, какие события есть в логах, как часто они встречаются. Отсортируем события по частоте

In [ ]:
data['event'].value_counts()

Посчитаем, сколько пользователей совершали каждое из этих событий, и долю пользователей, которые хоть раз совершали событие

In [ ]:
events_users=data.groupby('event').agg({'event':'count', 'device_id':'nunique'}).sort_values(by='device_id', ascending=False)
events_users.columns = ['event_count', 'users_count']
events_users = events_users.reset_index()
events_users['user_share'] = events_users['users_count'] / data['device_id'].nunique()
events_users

Скорее всего события происходят в следующем порядке:
* MainScreenAppear - появление главного экрана;
* Tutorial - руководство;
* OffersScreenAppear - появление экрана предложений;
* CartScreenAppear - появление экрана корзины;
* PaymentScreenSuccessful - появление экрана успешной оплаты.

Так как `Tutorial` имеет наименьшее кол-во пользователей, совершившее событие, можем предположить, что обучение не является обязательным и не влияет на то, дойдет ли пользователь до финальной покупки или нет

In [ ]:
#убираем этап Tutorial из дальшейшего анализа 
events_users = events_users[events_users['event'] != 'Tutorial']

In [ ]:
#строим воронку событий
fig = go.Figure(go.Funnel(x = events_users['users_count'], y = events_users['event'],
                         marker = {'color':'darkmagenta'},
                         textinfo = 'value+percent initial+percent previous'))
fig.layout.title='Воронка событий'
fig.show()

Ко 2 шагу - выбору товаров переходят 62% пользователей от общего числа, половина пользователей доходят до третьего шага и до оплаты заказа доходит лишь 48% пользователей от общего числа.

Воронка с пошаговой конверсией показывает, что между первым и вторым шагом мы теряем больше всего пользователей - 62% от тех, кто был на главном экране. 81% пользователей от предудыщего шага переходят в корзину и почти все из них оплачивают заказ (95%)

In [ ]:
share = (events_users._get_value(3,'users_count') / events_users._get_value(0,'users_count')).round(2)
print('{:.0%} пользователей доходит от первого события до оплаты'.format(share))

**Вывод**

1. Воронка событий идет в следующем порядке:
    - `MainScreenAppear` (появление главного экрана) - зашли на сайт
    - `Tutorial` (руководство) - изучали инструкцию
    - `OffersScreenAppear` (появление экрана предложений) - искали товар по каталогу
    - `CartScreenAppear` (появление экрана корзины) - добавили в корзину
    - `PaymentScreenSuccessful` (появление экрана успешной оплаты) - оплатили;
2. Событие `Tutorial` не несет особой значимости, поэтому его можно не принимать во внимание;
3. На второй этап (`OffersScreenAppear`) попадает 62% пользователей, на этом этапе мы теряем больше всего пользователей;
4. Полностью воронку событий проходит 48% пользователей

## Шаг 5. Изучение результатов эксперимента

Смотрим кол-во пользователей в каждой экспериментальной группе

In [ ]:
data = data.query('event != "Tutorial"')
exp_group = data.groupby('exp_id')['device_id'].nunique()
exp_group['246+247'] = exp_group[246] + exp_group[247]
exp_group

В каждой группе почти равное кол-во пользователей

Оценим 2 контрольные группы А/А-эксперимента, чтобы проверить корректность всех механизмов и расчётов.
И проверим, находят ли статистические критерии разницу между выборками 246 и 247

In [ ]:
#число пользователей в каждой группе
exp_246 = (data[data['exp_id'] == 246]['device_id']).nunique()
exp_247 = (data[data['exp_id'] == 247]['device_id']).nunique()
print('Количество пользователей в группах 246-247 различается на', round(100 - (exp_246 / exp_247 * 100), 2), '%')

Между выборками 246 и 247 разница чуть больше 1%

Выберем самое популярное событие

In [ ]:
#создадим сводную таблицу по событиям, с количество пользователей в разных группах и их долями в этих событиях
data_test = data.pivot_table(
    index='event', 
    columns='exp_id',
    values='device_id',
    aggfunc='nunique').reset_index().sort_values(by=246, ascending=False)

data_test['246+247'] = data_test[246]+data_test[247]
data_test['share_246'] = (data_test[246] / exp_group[246] * 100).round(2)
data_test['share_247'] = (data_test[247] / exp_group[247] * 100).round(2)
data_test['share_248'] = (data_test[248] / exp_group[248] * 100).round(2)

data_test

In [ ]:
#визуализируем 
fig = go.Figure()

fig.add_trace(go.Funnel(
    name = '246',
    y = data_test['event'],
    x = data_test[246],
    textinfo = 'value+percent initial'))

fig.add_trace(go.Funnel(
    name = '247',
    y = data_test['event'],
    x = data_test[247],
    textinfo = 'value+percent initial'))

fig.add_trace(go.Funnel(
    name = '248',
    y = data_test['event'],
    x = data_test[248],
    textinfo = 'value+percent initial'))

fig.layout.title='Воронка событий'
fig.show()

Самое популяпное событие - это `MainScreenAppear`(появление главного экрана). Данных всех трех групп очень близки по значениям

Перед началом А/В теста, проведем А/А тест. Он похож на A/B-тест, только группам показывают не разные, а одинаковые версии страниц. Если трафик и инструмент проведения A/A-теста не подвели, различий в показателях не будет. Также А/А-тест помогает определить длительность теста и методику анализа данных.

Критерии успешного A/A-теста:

- Количество пользователей в различных группах различается не более, чем на 1%;
- Для всех групп фиксируют и отправляют в системы аналитики данные об одном и том же;
- Различие ключевых метрик по группам не превышает 1% и не имеет статистической значимости;
- Попавший в одну из групп посетитель остаётся в этой группе до конца теста. Если пользователь видит разные версии исследуемой страницы в ходе одного исследования, неизвестно, какая именно повлияла на его решения. Значит, и результаты такого теста нельзя интерпретировать однозначно.

При помощи Z-теста (z-критерий Фишера), т.к. будем проводить сравнения двух независимых выборок большого размера, проверим гипотезу о равенстве долей, критерий значимости берем 1%, т.к наша цель – не увидеть различий в показателях

<u>Сформулируем гипотезы:</u>

    Н0 - Между долями в группах нет значимой разницы;
    Н1 - Между долями в группах есть значимая разница.

In [ ]:
#напишем функцию для проверки гипотез
def z_test(group1, group2, alpha):
    for i in data_test.index:
        # пропорция успехов в группах:
        p1 = data_test[group1][i] / exp_group[group1]
        p2 = data_test[group2][i] / exp_group[group2]
        p_combined = ((data_test[group1][i] + data_test[group2][i]) / 
                      (exp_group[group1] + exp_group[group2]))
        
        # разница пропорций в датасетах
        difference = p1 - p2
        # считаем статистику в ст.отклонениях стандартного нормального распределения
        z_value = difference / mth.sqrt(p_combined * (1 - p_combined) * 
                                        (1/exp_group[group1] + 1/exp_group[group2]))
        
        # задаем стандартное нормальное распределение (среднее 0, ст.отклонение 1)
        distr = st.norm(0, 1) 
        p_value = (1 - distr.cdf(abs(z_value))) * 2
        
        print('Cобытие: {}, p-значение: {p_value:.2f}'.format(data_test['event'][i], p_value=p_value))
        
        if (p_value < alpha):
            print('Отвергаем нулевую гипотезу: между долями есть значимая разница')
        else:
            print('Не получилось отвергнуть нулевую гипотезу, нет оснований считать доли разными')
        print('')

<u>Сравним тестовые группы 246 и 247</u>

In [ ]:
#передадим в функцию группы и критерий значимости
z_test(246, 247, 0.01)

В результате А/А теста видим, что не получилось отвергнуть нулевую гипотезу, т.е. нет статистически значимой разницы для всех событий воронки в двух контрольных группах.
Разделение на группы работает корректно. Значит А/А тест можно признать успешными и перейти к А/В тестированию

Для А/В тестирования примем стандартный уровень значимости равный 5% (т.к. не нужна высокая точность). Сравнивать будем группы 246/248 , 247/248 , 246+247/248, т.е.  проведем 3 теста по 4 событиями.

<u>Сформулируем гипотезы:</u>

    Н0 - Между долями в группах нет значимой разницы;
    Н1 - Между долями в группах есть значимая разница.

<u>Сравним группу 248 с тестовой группой 246</u>

In [ ]:
z_test(246, 248, 0.05)

<u>Сравним группу 248 с тестовой группой 247</u>

In [ ]:
z_test(247, 248, 0.05)

<u>Сравним группу 248 с совокупностью тестовых групп 246+247</u>

In [ ]:
z_test('246+247', 248, 0.05)

Во всех тестах не получилось отвергнуть нулевую гипотезу, т.е значимой разницы между группами не выявлено.
Тестирование прошло успешно. Однако изменение шрифта никак не повлияло на поведение пользователей

Проводили один А/А тест по 4 события и три А/В теста по 4 событиями, т.е кол-во проверок статистических гипотез = 16. Кроме поправки Бонферрони можем использовать метод Шидака для того, чтобы повысить мощность теста, сохраняя FWER < alpha. Формула для расчета: 1 - (1 - 0.05)**(1/16).


In [ ]:
#для примера применем метод Шидака для групп 248 и 246, где alpha = 0.05
z_test(246, 248, 1 - (1 - 0.05)**(1/16))

In [ ]:
#и применем метод Шидака для групп 246 и 247, где alpha = 0.01
z_test(246, 247, 1 - (1 - 0.05)**(1/16))

Результаты тестов не изменились 


**Вывод**

- За исследуемый период (с 2019-08-01 по 2019-08-07) приложением воспользовалось 7534 пользователя. 
- 48% пользователей от общего кол-ва доходит до покупки.
- Самый большой процент потери пользователей (38%) происходит на этапе появление экрана предложений. 
- По результатам A/A/B эксперимента - изменение шрифта никак не повлияло на поведение пользователей.